In [1]:
import os
import re
import warnings
import random
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import requests

from tqdm import tqdm
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE, ADASYN, SVMSMOTE, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.ensemble import BalancedRandomForestClassifier

from fairlearn.metrics import MetricFrame, demographic_parity_difference, equalized_odds_difference

from sentence_transformers import SentenceTransformer, util, CrossEncoder

import torch

from utils.feature_engineering import *
from utils.data_cleaning import *

warnings.filterwarnings('ignore')

ImportError: numpy.core.multiarray failed to import

## Training

In [ ]:
columns_to_keep = [
    "Sex_int",
    "Protected category",
    "Overall",
    "Technical Skills",
    "Standing/Position",
    "Comunication",
    "Maturity",
    "Dynamism",
    "Mobility",
    "English",
    "Hired",
    "Italian Residence",
    "European Residence",
    "Age Range_int",
    "experience_match_score",
    "Years Experience_int",
    "Years Experience.1_int",
    "current_salary_fit_score",
    "Current Ral",
    "Expected Ral",
    "Minimum Ral",
    "Ral Maximum",
    "expected_salary_fit_score",
    "study_title_score",
    "Study Level_int",
    "Study Title_int",
    "professional_similarity_score",
    "study_area_score",
    "general_similarity_score",
    "general_similarity_score_tfidf",
    "general_similarity_score_cross",
    "number_of_searches",
    "Distance Residence - Akkodis HQ",
    "Distance Residence - Assumption HQ",
]

#### Load Cleaned and Full Dataset

In [ ]:
df_cleaned = pd.read_csv('cleaned_dataset.csv')
dataset = pd.read_csv('full_dataset.csv')

### Models Comparison

In [ ]:
random_state = 42

df = df_cleaned[columns_to_keep].copy()
X = df.drop(columns=['Hired'])
y = df['Hired']

bool_cols = X.select_dtypes(include='bool').columns
non_bool_cols = [c for c in X.columns.difference(bool_cols) if c != 'Sex_int']
X[bool_cols] = X[bool_cols].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=random_state)
scaler = StandardScaler()
X_train[non_bool_cols] = scaler.fit_transform(X_train[non_bool_cols])
X_test[non_bool_cols] = scaler.transform(X_test[non_bool_cols])
imputer = SimpleImputer(strategy='mean')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

X_train_upsampled, y_train_upsampled = ADASYN(random_state=random_state).fit_resample(X_train_imputed, y_train)
downsampler_impl = RandomUnderSampler(sampling_strategy='majority', random_state=random_state)
X_train_downsampled, y_train_downsampled = downsampler_impl.fit_resample(X_train_imputed, y_train)

models = {
    'RandomForest': lambda: RandomForestClassifier(class_weight='balanced', random_state=random_state, max_depth=10, min_samples_split=5, n_estimators=100),
    'HistGradientBoosting': lambda: HistGradientBoostingClassifier(random_state=random_state),
    'XGBoost': lambda: XGBClassifier(scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(), random_state=random_state, eval_metric='logloss', max_depth=6),
    'LightGBM': lambda: LGBMClassifier(class_weight='balanced', random_state=random_state, max_depth=6, min_data_in_leaf=20, verbosity=-1),
    'LogisticRegression': lambda: LogisticRegression(class_weight='balanced', max_iter=1000, penalty='l2', C=0.1, solver='liblinear', random_state=random_state),
    'CatBoost': lambda: CatBoostClassifier(auto_class_weights='Balanced', silent=True, random_state=random_state, l2_leaf_reg=3, iterations=500, depth=6, learning_rate=0.05),
    'BalancedRF': lambda: BalancedRandomForestClassifier(random_state=random_state)
}

ensemble = lambda: VotingClassifier(
    estimators=[
        ('xgb', models['XGBoost']()),
        ('lgbm', models['LightGBM']()),
        ('cat', models['CatBoost']()),
        ('hist', models['HistGradientBoosting']()),
        ('brf', models['BalancedRF']())
    ],
    voting='soft'
)
models['Ensemble'] = ensemble

results_train = {}
results_test = {}

for strategy_name in ['Downsample', 'Original', 'Upsample']:
    strategy_results_train = []
    strategy_results_test = []
    
    for model_name, model in models.items():
        model = model()
        if strategy_name == 'Original':
            if model_name in ['XGBoost', 'LightGBM', 'CatBoost']:
                X_tr = X_train
                y_tr = y_train
                X_te = X_test
            else:
                X_tr = X_train_imputed
                y_tr = y_train
                X_te = X_test_imputed
        elif strategy_name == 'Upsample':
            X_tr = X_train_upsampled
            y_tr = y_train_upsampled
            X_te = X_test_imputed
        elif strategy_name == 'Downsample':
            X_tr = X_train_downsampled
            y_tr = y_train_downsampled
            X_te = X_test_imputed  
    
        model.fit(X_tr, y_tr)
        y_pred_train = model.predict(X_tr)
        y_pred_test = model.predict(X_te)
        
        strategy_results_train.append([
            model_name,
            f1_score(y_tr, y_pred_train),
            accuracy_score(y_tr, y_pred_train),
            precision_score(y_tr, y_pred_train),
            recall_score(y_tr, y_pred_train)
        ])
        
        strategy_results_test.append([
            model_name,
            f1_score(y_test, y_pred_test),
            accuracy_score(y_test, y_pred_test),
            precision_score(y_test, y_pred_test),
            recall_score(y_test, y_pred_test)
        ])
    
    results_train[strategy_name] = pd.DataFrame(strategy_results_train, columns=['Model', 'F1 Score', 'Accuracy', 'Precision', 'Recall'])
    results_test[strategy_name] = pd.DataFrame(strategy_results_test, columns=['Model', 'F1 Score', 'Accuracy', 'Precision', 'Recall'])

print("\nTraining Data Results:")
for strategy_name, result_df in results_train.items():
    print(f"\nResults for {strategy_name} Sampling (Training Data):")
    print(result_df.sort_values(by='F1 Score', ascending=False).to_string(index=False))

print("\nTest Data Results:")
for strategy_name, result_df in results_test.items():
    print(f"\nResults for {strategy_name} Sampling (Test Data):")
    print(result_df.sort_values(by='F1 Score', ascending=False).to_string(index=False))

strategy_colors = {
    'Original': sns.color_palette("tab10")[0],
    'Upsample': sns.color_palette("tab10")[1],
    'Downsample': sns.color_palette("tab10")[2],
}

plt.figure(figsize=(12, 8))

for strategy_name, result_df in results_test.items():
    color = strategy_colors[strategy_name]
    plt.plot(result_df['Model'], result_df['F1 Score'], label=f"{strategy_name} (Test)", marker='o', linestyle='--', color=color)

for strategy_name, result_df in results_train.items():
    color = strategy_colors[strategy_name]
    plt.plot(result_df['Model'], result_df['F1 Score'], label=f"{strategy_name} (Train)", marker='o', linestyle='-', color=color)

plt.title('F1 Score Comparison Across Strategies (Train vs Test)')
plt.ylabel('F1 Score')
plt.xlabel('Models')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


#### Summary

- **Ensemble** consistently outperforms other models on **test data** across all sampling strategies, achieving the **highest F1 scores** (up to 0.80 with original dataset).

- **LightGBM**, **HistGradientBoosting**, **XGBoost** and **CatBoost** models also perform well, particularly with original and upsampled data.

- **RandomForest** and **BalancedRF** show solid performance but slightly trail behind the top models.

- **BalancedRF** benefits most from upsampling but underperforms with original and downsampled data.

- **Logistic Regression** lags across all settings, confirming that more complex models handle the task better.

> All models show perfect or near-perfect results on training data, indicating **clear overfitting**, likely due to the **limited dataset size**.


### Features Removal Comparison

In [ ]:
def compare_feature_sets(feature_sets, test_models):
    results = []

    for feat_set_name, feat_cols in tqdm(feature_sets.items(), desc='Feature sets'):
        X_sub = df_cleaned[feat_cols].copy()
        y_sub = df_cleaned['Hired']

        bool_cols = X_sub.select_dtypes(include='bool').columns
        non_bool_cols = X_sub.columns.difference(bool_cols)

        X_sub[bool_cols] = X_sub[bool_cols].astype(int)

        X_train, X_test, y_train, y_test = train_test_split(
            X_sub, y_sub, stratify=y_sub, test_size=0.2, random_state=random_state
        ) 
        scaler = StandardScaler()
        X_train[non_bool_cols] = scaler.fit_transform(X_train[non_bool_cols])
        X_test[non_bool_cols] = scaler.transform(X_test[non_bool_cols])
        imputer = SimpleImputer(strategy='mean')
        X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
        X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

        for model_name, model_lambda in test_models.items():
            model = model_lambda()
            if model_name in ['LightGBM', 'CatBoost']:
                X_tr = X_train
                X_te = X_test
            else:
                X_tr = X_train_imputed
                X_te = X_test_imputed
            model.fit(X_tr, y_train)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)
            f1_test = f1_score(y_test, y_pred_test)
            f1_train = f1_score(y_train, y_pred_train)
            results.append({
                'Feature Set': feat_set_name,
                'Model': model_name,
                'Train F1': f1_train,
                'Test F1': f1_test
            })

    results_df = pd.DataFrame(results)

    best_row = results_df.loc[results_df['Test F1'].idxmax()]
    print(f"\nBest performance:\nFeature Set: {best_row['Feature Set']}, Model: {best_row['Model']}, F1 Score: {best_row['Test F1']:.4f}")
    return results_df

In [ ]:
feature_columns = [col for col in columns_to_keep if col != 'Hired']

feature_sets = {
    'general_similarity_score_cross': [col for col in feature_columns if col not in ['general_similarity_score', 'general_similarity_score_tfidf']],
    'general_similarity_score': [col for col in feature_columns if col not in ['general_similarity_score_tfidf', 'general_similarity_score_cross']],
    'general_similarity_score_tfidf': [col for col in feature_columns if col not in ['general_similarity_score', 'general_similarity_score_cross']],
    'None ':  [col for col in feature_columns if col not in ['general_similarity_score_tfidf', 'general_similarity_score', 'general_similarity_score_cross']], 
    'all': feature_columns,
    }

test_models = {
    "LightGBM":models['LightGBM'],
    'CatBoost': models['CatBoost'],
    'Ensemble': models['Ensemble'],
}

results_df = compare_feature_sets(feature_sets, test_models)
results_df

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=results_df, x="Feature Set", y="Test F1", hue="Model")
plt.xticks(rotation=45, ha="right")
plt.ylim(0.8*min(results_df['Test F1']))
plt.title("F1 Scores by Feature Set and Model")
plt.tight_layout()
plt.legend(title="Model")
plt.grid(axis='y')

plt.show()

#### Feature Set Comparison Summary

The best performance (F1 = 0.796) was achieved using **only the `general_similarity_score` feature** with the Ensemble model. This suggests that deep semantic similarity captured through a bi-encoder model can provide a highly effective signal for predicting hires, even if it's not consistently the best across all models.

- **`general_similarity_score`** uses a SentenceTransformer encoder and cosine similarity to capture deep semantic relationships.
- **`general_similarity_score_cross`** leverages a cross-encoder transformer that jointly processes candidate and job texts, allowing for nuanced contextual matching.
- **`general_similarity_score_tfidf`** calculates cosine similarity between TF-IDF vectors, highlighting surface-level lexical overlap.

Both the `general_similarity_score_cross` and `all` feature sets also produced **strong and consistent results** across multiple models, demonstrating the robustness of combining similarity signals. In particular, `general_similarity_score_cross` often yielded reliable performance, though it did **not outperform** the peak result from `general_similarity_score` with the Ensemble model.

Notably, excluding all similarity features leads to a significant drop in F1 (e.g., Ensemble F1 falls to 0.775), emphasizing the importance of text similarity for this task.

Given that `general_similarity_score` achieves the **highest single F1 score**, even if not consistently the best across all settings, we opt to use **only `general_similarity_score`** in the final model. This choice balances performance with simplicity and computational efficiency, while capturing the strongest individual signal observed.


In [ ]:
feature_sets = {
    "base_attributes": [
        "Sex_int",
        "Protected category",
        "Overall",
        "Technical Skills",
        "Standing/Position",
        "Comunication",
        "Maturity",
        "Dynamism",
        "Mobility",
        "English",
        "Italian Residence",
        "European Residence",
        "Age Range_int",
        "Years Experience_int",
        "Years Experience.1_int",
        "Current Ral",
        "Expected Ral",
        "Minimum Ral",
        "Ral Maximum",
        "Study Level_int",
        "Study Title_int",
        "number_of_searches",
    ],
    "custom_scores": [
        "experience_match_score",
        "current_salary_fit_score",
        "expected_salary_fit_score",
        "study_title_score",
        "professional_similarity_score",
        "study_area_score",
        "general_similarity_score",
        "Distance Residence - Akkodis HQ",
        "Distance Residence - Assumption HQ",
    ],
    "custom_scores_with_essential_base_attributes": [
        # Base Attributes
        "Sex_int",
        "Protected category",
        "Overall",
        "Technical Skills",
        "Standing/Position",
        "Comunication",
        "Maturity",
        "Dynamism",
        "Mobility",
        "English",
        "Italian Residence",
        "European Residence",
        "Age Range_int",
        "number_of_searches",
        # Custom Similarity Scores
        "experience_match_score",
        "current_salary_fit_score",
        "expected_salary_fit_score",
        "study_title_score",
        "professional_similarity_score",
        "study_area_score",
        "general_similarity_score",
        "Distance Residence - Akkodis HQ",
        "Distance Residence - Assumption HQ",
    ],
    "base_attributes_with_essential_custom_scores": [
        # Base Attributes
        "Sex_int",
        "Protected category",
        "Overall",
        "Technical Skills",
        "Standing/Position",
        "Comunication",
        "Maturity",
        "Dynamism",
        "Mobility",
        "English",
        "Italian Residence",
        "European Residence",
        "Age Range_int",
        "number_of_searches",
        # Essential Custom Similarity Scores
        "study_area_score",
        "general_similarity_score",
        "Distance Residence - Akkodis HQ",
        "Distance Residence - Assumption HQ",
        # Additional Base Attributes
        "Years Experience_int",
        "Years Experience.1_int",
        "Current Ral",
        "Expected Ral",
        "Minimum Ral",
        "Ral Maximum",
        "Study Level_int",
        "Study Title_int",
    ],
    "all": [
        # Base Attributes
        "Sex_int",
        "Protected category",
        "Overall",
        "Technical Skills",
        "Standing/Position",
        "Comunication",
        "Maturity",
        "Dynamism",
        "Mobility",
        "English",
        "Italian Residence",
        "European Residence",
        "Age Range_int",
        "number_of_searches", 
        # Custom Similarity Scores
        "experience_match_score",
        "expected_salary_fit_score",
        "current_salary_fit_score",
        "professional_similarity_score",
        "study_area_score",
        "general_similarity_score",
        "study_title_score",
        "Distance Residence - Akkodis HQ",
        "Distance Residence - Assumption HQ",
        # Additional Base Attributes
        "Years Experience_int",
        "Years Experience.1_int",
        "Current Ral",
        "Expected Ral",
        "Minimum Ral",
        "Ral Maximum",
        "Study Level_int",
        "Study Title_int",  
    ],
}
results_df = compare_feature_sets(feature_sets, test_models)
results_df

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=results_df, x="Feature Set", y="Test F1", hue="Model")
plt.xticks(rotation=45, ha="right")
plt.ylim(0.8*min(results_df['Test F1']))
plt.title("F1 Scores by Feature Set and Model")
plt.tight_layout()
plt.legend(title="Model")
plt.grid(axis='y')

plt.show()

#### Feature Set Comparison Summary

The best performance is achieved with the **`all`** feature set using the **CatBoost model**, reaching a **Test F1 Score of 0.8079**.

Summary of the tested feature sets:

- **base_attributes**  
  Encompasses a comprehensive set of features describing both the **candidate** (e.g., age, residence, language, education, experience) and the **job requirements or evaluations** (e.g., technical skills, position, salary ranges, overall fit scores).  
  This foundational information captures the context of both parties but lacks direct indicators of compatibility between them. Performance is moderate as a result.

- **custom_scores**  
  Focuses exclusively on engineered features that **quantify the match** between candidate and job — including salary fit, study alignment, professional similarity, and location distance.  
  These scores alone underperform, suggesting that without descriptive context, match scores don't provide enough standalone signal.

- **custom_scores_with_essential_base_attributes**  
  Builds on the custom similarity scores by integrating **critical base attributes** that anchor the match in relevant candidate-job context (like key demographics and job-related features).  
  This hybrid set performs very well, offering a balance between match precision and contextual understanding.

- **base_attributes_with_essential_custom_scores**  
  Starts with a rich base attribute set and adds only the **most important custom scores** (e.g., general similarity, location distance).  
  It effectively reinforces candidate-job context with minimal additional complexity, also producing strong results.

- **all**  
  Combines **every available feature** into one comprehensive set — including all base attributes and all custom scores.  
  While this approach slightly edges out others in performance, the marginal gain may not justify the added feature redundancy and risk of overfitting.

> **Conclusion:** While the `all` feature set shows the highest F1 score, we select the **`custom_scores_with_essential_base_attributes`** feature set as the optimal choice. It provides nearly equivalent performance with fewer features, reducing complexity and redundancy. These results also highlight that:
> - **Base attributes alone** are not sufficient, as they lack direct match indicators.
> - **Custom similarity scores alone** are also insufficient, as they lack foundational context.
> - The best results come from combining **custom match scores** with **key descriptive features**, ensuring both **contextual grounding** and **match relevance**.


### Sampling Techniques

In [ ]:
df = df_cleaned[feature_sets['custom_scores_with_essential_base_attributes']+['Hired']].copy()
X = df.drop(columns=['Hired'])
y = df['Hired']

bool_cols = X.select_dtypes(include='bool').columns
non_bool_cols = X.columns.difference(bool_cols)

X[bool_cols] = X[bool_cols].astype(int)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=random_state
)
scaler = StandardScaler()
X_train[non_bool_cols] = scaler.fit_transform(X_train[non_bool_cols])
X_test[non_bool_cols] = scaler.transform(X_test[non_bool_cols])

imputer = SimpleImputer(strategy='mean')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

X_orig, y_orig = X_train_imputed, y_train

minority_count = sum(y_orig == 1)
majority_count = sum(y_orig == 0)

smote_ratios = [0.25, 0.5, 0.75, 1.0]

resampled_datasets = {}

resampled_datasets['Original'] = {'imputed': (X_orig, y_orig, X_test_imputed), 'original': (X_train, y_train, X_test)}

for ratio in smote_ratios:
    name = f'SMOTE_{int(ratio*100)}%'
    smote = SMOTE(sampling_strategy=ratio, random_state=random_state)
    X_res, y_res = smote.fit_resample(X_orig, y_orig)
    resampled_datasets[name] = (X_res, y_res)

for ratio in smote_ratios:
    name = f'ADASYN_{int(ratio*100)}%'
    adasyn = ADASYN(sampling_strategy=ratio, random_state=random_state)
    X_res, y_res = adasyn.fit_resample(X_orig, y_orig)
    resampled_datasets[name] = (X_res, y_res)

for ratio in smote_ratios:
    name = f'BorderlineSMOTE_{int(ratio*100)}%'
    bl_smote = BorderlineSMOTE(sampling_strategy=ratio, random_state=random_state)
    X_res, y_res = bl_smote.fit_resample(X_orig, y_orig)
    resampled_datasets[name] = (X_res, y_res)

for ratio in smote_ratios:
    name = f'SVMSMOTE_{int(ratio*100)}%'
    svm_smote = SVMSMOTE(sampling_strategy=ratio, random_state=random_state)
    X_res, y_res = svm_smote.fit_resample(X_orig, y_orig)
    resampled_datasets[name] = (X_res, y_res)

resampled_datasets['Downsampling'] = RandomUnderSampler(random_state=random_state).fit_resample(X_orig, y_orig)

results = []

for strategy_name, trainset in tqdm(resampled_datasets.items(), desc='Training...'):
    for model_name, model_lambda in test_models.items():
        if strategy_name == 'Original':
            if model_name in ['LightGBM', 'CatBoost']:
                X_tr, y_tr, X_te = trainset['original']
            else: 
                X_tr, y_tr, X_te = trainset['imputed']
        else:
            (X_tr, y_tr) = trainset
            X_te = X_test_imputed
        model = model_lambda()
        model.fit(X_tr, y_tr)
        y_pred_test = model.predict(X_te)
        y_pred_train = model.predict(X_tr)
        f1_test = f1_score(y_test, y_pred_test)
        f1_train = f1_score(y_tr, y_pred_train)
        results.append({
            'Resampling': strategy_name,
            'Model': model_name,
            'Train F1': f1_train,
            'Test F1': f1_test
        })

results_df = pd.DataFrame(results)
results_df

In [ ]:
min_f1 = results_df['Test F1'].min()

plt.figure(figsize=(16, 8))
sns.barplot(
    data=results_df,
    x='Resampling',
    y='Test F1',
    hue='Model',
    palette='Set2'
)
plt.title('F1 Score of CatBoost, LightGBM and  Ensemble Models Across Resampling Techniques & Ratios')
plt.xticks(rotation=90)
plt.ylabel('F1 Score')
plt.ylim(bottom=0.95 * min_f1)
plt.xlabel('Resampling Strategy')
plt.legend(title='Model')
plt.tight_layout()
plt.show()

#### Resampling Strategy Comparison Summary

This experiment compared various resampling techniques to mitigate class imbalance, evaluating their impact on **CatBoost**, **LightGBM**, and **Ensemble** models based on Train and Test F1 scores.

##### Best Performance
- **Top Result:** `SVMSMOTE_75%` with **Ensemble** (Test F1 = **0.8241**)

##### Final Choice: `SMOTE_75%`
We select **SMOTE with 75% ratio** as the preferred resampling strategy due to its **strong performance** and **balanced data augmentation**:

- **Robust Test F1 Scores**:
  - LightGBM F1 = **0.8223**
  - CatBoost F1 = **0.8000**
  - Ensemble F1 = **0.8241**
- **Improved Generalization** compared to lower SMOTE ratios (25%, 50%) and original data.
- **Controlled synthetic sample generation** avoids overfitting seen at 100% oversampling levels.

##### How Sampling Ratio Affects Performance

- **Original dataset (no resampling)** provides a solid baseline with moderate test F1 scores (LightGBM: 0.7671, CatBoost: 0.7902, Ensemble: 0.8000), but the class imbalance limits further improvement.
- **Lower SMOTE ratios (25%, 50%)** improve Test F1 compared to the original dataset by alleviating class imbalance but may still under-represent the minority class, limiting gains.
- **At 75% oversampling**, the minority class is better represented, leading to improved model generalization and higher F1 scores across models.
- **Increasing to 100% oversampling** tends to introduce noise and redundancy in synthetic samples, causing slight performance drops, especially visible in CatBoost and Ensemble results.
- More complex resampling techniques like **SVMSMOTE** and **BorderlineSMOTE** show competitive or better performance at some ratios but come with increased computational cost and complexity.

##### Key Observations

- **Original data** shows that imbalance restricts maximum achievable performance despite strong training F1 scores.
- **SMOTE_75%** strikes the best balance between addressing class imbalance and avoiding overfitting or noise.
- **SVMSMOTE_75%** achieves the highest overall Test F1 with the Ensemble but at greater complexity.
- **ADASYN** methods deliver reasonable but less consistent results.
- **Downsampling** drastically reduces test performance despite perfect training scores, indicating overfitting.

> **Conclusion:**  
We select **SMOTE with a 75% oversampling ratio** as the optimal resampling strategy. It effectively balances minority class representation and avoids overfitting, leading to strong and consistent predictive performance across models while maintaining manageable computational costs.


### Fairness Metrics

In [ ]:
df = df_cleaned.copy()

protected_attributes = [
    'Sex_int', 'Protected category', 'Age Range_int',
    'Italian Residence', 'European Residence'
]

df = df.dropna(subset=protected_attributes).reset_index(drop=True)

X = df.drop(columns=['Hired'])[feature_sets['custom_scores_with_essential_base_attributes']]
y = df['Hired']

bool_cols = X.select_dtypes(include='bool').columns
non_bool_cols = X.columns.difference(bool_cols)

X[bool_cols] = X[bool_cols].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=random_state)

scaler = StandardScaler()
X_train[non_bool_cols] = scaler.fit_transform(X_train[non_bool_cols])
X_test[non_bool_cols] = scaler.transform(X_test[non_bool_cols])

imputer = SimpleImputer(strategy='mean')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

svm_smote = SMOTE(sampling_strategy=0.75, random_state=random_state)
X_res, y_res = svm_smote.fit_resample(X_train_imp, y_train)

majority_count = (y_res == 0).sum()
minority_count = (y_res == 1).sum()

catboost_model = models['CatBoost']()
lightgbm_model = models['LightGBM']()
ensemble_model = models['Ensemble']()

catboost_model.fit(X_res, y_res)
lightgbm_model.fit(X_res, y_res)
ensemble_model.fit(X_res, y_res)


y_pred_cat = catboost_model.predict(X_test_imp)
y_pred_lgb = lightgbm_model.predict(X_test_imp)
y_pred_ens = ensemble_model.predict(X_test_imp)

performance_results = []
fairness_results = []

for model_name, y_pred in [('CatBoost', y_pred_cat), ('LightGBM', y_pred_lgb), ('Ensemble', y_pred_ens)]:
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    performance_results.append({
        'Model': model_name,
        'Precision': precision,
        'Recall': recall
    })
    
    for attr in protected_attributes:
        sensitive_features_test = dataset.loc[X_test.index, attr]
        dp_diff = demographic_parity_difference(
            y_true=y_test,
            y_pred=y_pred,
            sensitive_features=sensitive_features_test
        )
    
        eo_diff = equalized_odds_difference(
            y_true=y_test,
            y_pred=y_pred,
            sensitive_features=sensitive_features_test
        )
        
        fairness_results.append({
            'Model': model_name, 
            'Attribute': attr,
            'Demographic Parity Diff': dp_diff,
            'Equalized Odds Diff': eo_diff
        })


performance_df = pd.DataFrame(performance_results)
fairness_df = pd.DataFrame(fairness_results)

long_perf_df = pd.melt(
    performance_df,
    id_vars=['Model'],
    value_vars=['Precision', 'Recall'],
    var_name='Metric',
    value_name='Score'
)

In [ ]:
fairness_long = fairness_df.melt(
    id_vars=['Model', 'Attribute'],
    value_vars=['Demographic Parity Diff', 'Equalized Odds Diff'],
    var_name='Fairness Metric',
    value_name='Score'
)

for metric in ['Demographic Parity Diff', 'Equalized Odds Diff']:
    plt.figure(figsize=(14, 7))
    sns.barplot(
        data=fairness_df,
        x='Attribute',
        y=metric,
        hue='Model',
        palette='Set2',
        ci=None,
        dodge=True
    )
    plt.title(f"{metric} Across Attributes")
    plt.axhline(0, linestyle='--', color='gray')
    plt.ylabel("Difference (Ideal = 0)")
    plt.xticks(rotation=45)
    plt.legend(title='Model')
    plt.tight_layout()
    plt.show()

for attr in protected_attributes:
    plt.figure(figsize=(14, 6))
    
    combined_data = []
    
    for model_name, y_pred in [('CatBoost', y_pred_cat),('LightGBM', y_pred_lgb), ('Ensemble', y_pred_ens)]:
        sensitive_features_test = df.loc[X_test.index, attr]
        
        mf = MetricFrame(
            metrics={'Precision': precision_score, 'Recall': recall_score},
            y_true=y_test,
            y_pred=y_pred,
            sensitive_features=sensitive_features_test
        )
        
        per_group_metrics = mf.by_group.reset_index()
        per_group_metrics['Model'] = model_name
        combined_data.append(per_group_metrics)
    
    combined_df = pd.concat(combined_data)
    
    combined_melted = combined_df.melt(
        id_vars=[attr, 'Model'],
        value_vars=['Precision', 'Recall'],
        var_name='Metric',
        value_name='Score'
    )
    
    combined_melted['Model_Metric'] = combined_melted['Model'] + ' | ' + combined_melted['Metric']
    
    sns.barplot(
        data=combined_melted,
        x=attr,
        y='Score',
        hue='Model_Metric',
        palette='Set2'
    )
    
    plt.title(f'Precision and Recall by Group for {attr}')
    plt.ylim(0, 1)
    plt.ylabel('Score')
    plt.xlabel(attr)
    plt.xticks(rotation=45)
    plt.legend(title='Model | Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()



In [ ]:
fairness_df

In [ ]:
performance_df

#### Preliminary Analysis of Model Performance and Fairness with Protected Attributes

##### Predictive Performance
- The **CatBoost** model achieves the highest precision (0.80), while the **Ensemble** model leads in recall (0.83).
- **LightGBM** performs comparably, with balanced precision (0.79) and recall (0.81).
- Overall, the models demonstrate strong predictive ability when trained with protected attributes included.

##### Fairness Metrics
- **Sex_int:** All models show low demographic parity differences (~0.02–0.04) and low to moderate equalized odds differences (~0.03–0.09), indicating relatively fair treatment across sex groups.
- **Protected category:** Exhibits consistently high demographic parity differences (~0.22–0.23) but low equalized odds differences (~0.03–0.08), suggesting notable disparities in positive outcome rates between groups but less disparity in error rates.
- **Age Range_int:** Moderate demographic parity differences (~0.10–0.12) with more pronounced equalized odds differences for CatBoost (0.60) and LightGBM/Ensemble (0.40), pointing to some fairness concerns regarding age.
- **Italian Residence:** Shows moderate to high demographic parity differences (~0.09–0.12) and high equalized odds differences (~0.78–0.84), indicating potential geographic bias and error rate disparities.
- **European Residence:** Demographic parity differences vary widely—from very low (0.003 in LightGBM) to moderate (~0.10–0.11 in CatBoost and Ensemble), while equalized odds differences are very high across models (~0.78–0.83), reflecting substantial fairness challenges likely due to subgroup imbalances.

##### Next Steps
To mitigate these fairness concerns, we will experiment with **removing protected attributes** from the training data. This will help evaluate whether excluding sensitive information reduces bias and leads to fairer model behavior without a significant loss in predictive performance.


### Removing Protected Atttributes

In [ ]:
df = df_cleaned.copy()

protected_attributes = [
    'Sex_int', 'Protected category', 'Age Range_int',
    'Italian Residence', 'European Residence'
]

df = df.dropna(subset=protected_attributes).reset_index(drop=True)

feature_sets_dict = {
    'With Protected': feature_sets['custom_scores_with_essential_base_attributes'],
    'Without Protected': list(set(feature_sets['custom_scores_with_essential_base_attributes']) - set(protected_attributes))
}

results = []
pergroup_all = []

for setting_label, features in feature_sets_dict.items():
    X = df[features].copy()
    y = df['Hired']

    bool_cols = X.select_dtypes(include='bool').columns
    non_bool_cols = X.columns.difference(bool_cols)
    X[bool_cols] = X[bool_cols].astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=random_state)
    scaler = StandardScaler()
    X_train[non_bool_cols] = scaler.fit_transform(X_train[non_bool_cols])
    X_test[non_bool_cols] = scaler.transform(X_test[non_bool_cols])

    imputer = SimpleImputer(strategy='mean')
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
    X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

    X_res, y_res = SMOTE(sampling_strategy=0.5, random_state=random_state).fit_resample(X_train_imp, y_train)

    catboost_model = models['CatBoost']()
    lightgbm_model = models['LightGBM']()
    ensemble_model = models['Ensemble']()

    catboost_model.fit(X_res, y_res)
    lightgbm_model.fit(X_res, y_res)
    ensemble_model.fit(X_res, y_res)

    y_pred_cat = catboost_model.predict(X_test_imp)
    y_pred_lgb = lightgbm_model.predict(X_test_imp)
    y_pred_ens = ensemble_model.predict(X_test_imp)

    for model_name, y_pred in [('CatBoost', y_pred_cat), ('LightGBM', y_pred_lgb), ('Ensemble', y_pred_ens)]:
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        results.append({
            'Setting': setting_label,
            'Model': model_name,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1
        })

        for attr in protected_attributes:
            sensitive_features_test = df.loc[X_test.index, attr]

            dp_diff = demographic_parity_difference(y_test, y_pred, sensitive_features=sensitive_features_test)
            eo_diff = equalized_odds_difference(y_test, y_pred, sensitive_features=sensitive_features_test)

            results.append({
                'Setting': setting_label,
                'Model': model_name,
                'Attribute': attr,
                'Metric': 'Demographic Parity Diff',
                'Score': dp_diff
            })
            results.append({
                'Setting': setting_label,
                'Model': model_name,
                'Attribute': attr,
                'Metric': 'Equalized Odds Diff',
                'Score': eo_diff
            })

            mf = MetricFrame(
                metrics={'precision': precision_score, 'recall': recall_score},
                y_true=y_test,
                y_pred=y_pred,
                sensitive_features=sensitive_features_test
            )

            pergroup_all.append(pd.DataFrame({
                'Setting': setting_label,
                'Model': model_name,
                'Attribute': attr,
                'Group': mf.by_group.index,
                'Precision': mf.by_group['precision'].values,
                'Recall': mf.by_group['recall'].values
            }))

performance_df = pd.DataFrame([r for r in results if 'F1 Score' in r])
fairness_df = pd.DataFrame([r for r in results if 'Attribute' in r])
pergroup_df = pd.concat(pergroup_all, ignore_index=True)


In [ ]:
performance_df

In [ ]:
fairness_df

In [ ]:
pergroup_df

In [ ]:
fairness_df['Model_Setting'] = fairness_df['Model'] + ' | ' + fairness_df['Setting']

plt.figure(figsize=(18, 7))

fairness_df['Attribute_Model'] = fairness_df['Attribute'] + ' | ' + fairness_df['Model']

metrics = fairness_df['Metric'].unique()

for metric in metrics:
    plt.figure(figsize=(18, 7))
    subset = fairness_df[fairness_df['Metric'] == metric]

    order = sorted(subset['Attribute_Model'].unique(), key=lambda x: x.split(' | ')[0])
    
    sns.barplot(
        data=subset,
        x='Attribute_Model',
        y='Score',
        hue='Setting',          
        palette='Set2',
        ci=None,
        order=order
    )
    
    plt.title(f"Fairness Metric: {metric} (With vs Without Protected Attributes)")
    plt.axhline(0, linestyle='--', color='gray')
    plt.xticks(rotation=75, ha='right')
    plt.ylabel('Score')
    plt.xlabel('Attribute | Model')
    plt.legend(title='Setting')
    plt.tight_layout()
    plt.show()


In [ ]:
performance_df['Model+Setting'] = performance_df['Model'] + ' | ' + performance_df['Setting']
fairness_df['Model+Setting'] = fairness_df['Model'] + ' | ' + fairness_df['Setting']
pergroup_df['Model+Setting'] = pergroup_df['Model'] + ' | ' + pergroup_df['Setting']

plt.figure(figsize=(10, 6))
melted_perf = performance_df.melt(
    id_vars=['Model+Setting'], 
    value_vars=['Precision', 'Recall', 'F1 Score'],
    var_name='Metric', 
    value_name='Value'
)
sns.barplot(data=melted_perf, x='Model+Setting', y='Value', hue='Metric', palette='pastel')
plt.title("Model Performance: With vs Without Protected Attributes")
plt.xticks(rotation=45)
plt.ylim(0.6, 1)
plt.tight_layout()
plt.show()



for attr in protected_attributes:
    df_attr = pergroup_df[pergroup_df['Attribute'] == attr]

    for metric in ['Precision', 'Recall']:
        plt.figure(figsize=(12, 6))
        sns.barplot(
            data=df_attr,
            x='Group',
            y=metric,
            hue='Model+Setting',
            palette='Set2'
        )
        plt.title(f"{metric} by Group for '{attr}': With vs Without Protected Attributes")
        plt.ylim(0, 1)
        plt.xticks(rotation=45)
        plt.legend(title='Model + Setting')
        plt.tight_layout()
        plt.show()




#### Analysis Summary: Impact of Including Protected Attributes on Model Performance and Fairness

##### Predictive Performance
Including protected attributes **consistently improves model performance** across all three models:

- **CatBoost** saw an increase in precision from **0.780 → 0.806**, and in F1 score from **0.804 → 0.823**.
- **LightGBM** maintained similar precision (**0.780**) but improved slightly in recall.
- **Ensemble** benefited the most, with precision rising to **0.784** and F1 score to **0.816**, making it the top performer in the "With Protected" setting.

Overall, incorporating protected features allows models to better capture patterns associated with subgroup characteristics, boosting predictive accuracy and balance.

---

##### Fairness Metrics

**Demographic Parity Difference (DPD)**

- There is **no systematic reduction or increase** in DPD when protected attributes are **omitted**, even though the goal of omission was to improve fairness.
- Changes vary significantly by model and group:
  - *CatBoost → Age Range*: DPD increases from **0.135 (Without)** to **0.168 (With)**.
  - *LightGBM → Protected category*: DPD stays consistently low (**0.013**) in both settings.

**Conclusion**: Although we expected that **omitting protected attributes would reduce bias**, the results show **no consistent improvement in demographic parity**. Impacts are **highly model- and attribute-dependent**.

**Equalized Odds Difference (EOD)**

- Similarly, **no clear trend emerges** when protected attributes are removed:
  - *CatBoost → Protected category*: EOD increases slightly from **0.839 (Without)** to **0.849 (With)**.
  - *Ensemble → European Residence*: EOD also increases from **0.819 (Without) → 0.851 (With)**.
  - *Ensemble → Sex_int*: EOD improves, decreasing from **0.058 (Without) → 0.017 (With)**.

**Conclusion**: Despite the intention that **removing protected attributes might improve equalized odds**, the data shows **no systematic benefit**. In some cases, disparities even worsen, reinforcing that **omission alone is not a reliable fairness strategy**.

---

##### Key Takeaways
- **Including protected attributes improves overall accuracy and helps mitigate demographic parity gaps.**
- **Equalized odds disparities persist**, especially for sensitive attributes like “Protected category” and “European Residence”.
- **Ensemble model** consistently outperforms others in balancing performance and fairness when protected attributes are used.
- **Omitting protected features** generally results in **slightly lower or unchanged F1 scores**, and **does not lead to consistent improvements in fairness metrics**. In some cases, fairness disparities even **increase**, indicating that omission is **not a reliable fairness strategy**.
